# 01 – Preparación de datos

- **Proyecto:** Evolución de los ingresos laborales según edad, educación y región en Argentina: antes, durante y después de la cuarentena.
- **Objetivo:** Cargar, verificar y unificar las bases individuales de la EPH; definir la población de estudio y las variables derivadas, y expresar los ingresos en pesos de octubre de 2026.
- **Entradas:** Los archivos zip de la EPH y el archivo `serie_ipc_divisiones.csv`, descargados del sitio del INDEC.
- **Orden de ejecución:** Se ejecuta antes que `02_analisis`. Necesita conexión a internet la primera vez. Si la sesión se reinicia, se deben volver a ejecutar las secciones 1 y 2.

### Salidas
| Archivo | Contenido |
|---|---|
| `datos_originales/eph/*.zip` | Bases originales |
| `documentacion/registro_descarga_eph.csv` | Registro de descarga EPH |
| `documentacion/verificacion_bases_eph.csv` | Verificación de columnas |
| `datos_procesados/eph_t2_2017_2026_18mas.parquet` | Base unificada |
| `datos_originales/ipc/serie_ipc_divisiones.csv` | Archivo IPC original |
| `documentacion/registro_descarga_ipc.csv` | Registro de descarga IPC |
| `datos_procesados/deflactor_ipc_t2.csv` | Deflactor calculado |
| `datos_procesados/eph_t2_2017_2026_18mas_ingresos.parquet` | Base final con ingresos |

Las secciones 3 a 6 corresponden a la Etapa 2 (preparar y limpiar los datos) y las secciones 7 a 11 a la Etapa 3 (preparar los ingresos).

**Índice de secciones:**
1. Configuración del entorno
2. Librerías y versiones del entorno
3. Descarga de las bases de la EPH
4. Lectura y verificación de las bases
5. Unión de los diez años y control de calidad
6. Población de estudio y variables derivadas
7. Situación del ingreso entre los adultos ocupados
8. Decisiones sobre los ingresos
9. IPC nacional y deflactor
10. Ingresos en pesos de octubre de 2026
11. Guardado de las bases finales
12. Resumen y limitaciones

## 1. Configuración del entorno
Montaje de Google Drive, definición de las rutas del proyecto y verificación de que todas las carpetas existen.

In [1]:
from google.colab import drive   # trae la herramienta de Colab para conectar Drive
drive.mount('/content/drive')    # monta tu Drive en la ruta /content/drive

Mounted at /content/drive


In [2]:
from pathlib import Path

RAIZ = Path('/content/drive/MyDrive/eph_ingresos')
DATOS_ORIGINALES = RAIZ / 'datos_originales'
DATOS_EPH        = DATOS_ORIGINALES / 'eph'
DATOS_IPC        = DATOS_ORIGINALES / 'ipc'
DATOS_PROCESADOS = RAIZ / 'datos_procesados'
RESULTADOS       = RAIZ / 'resultados'
DOCUMENTACION    = RAIZ / 'documentacion'

rutas = [RAIZ, DATOS_ORIGINALES, DATOS_EPH, DATOS_IPC, DATOS_PROCESADOS, DOCUMENTACION, RESULTADOS]

for ruta in rutas:
    print(ruta, ruta.exists())

/content/drive/MyDrive/eph_ingresos True
/content/drive/MyDrive/eph_ingresos/datos_originales True
/content/drive/MyDrive/eph_ingresos/datos_originales/eph True
/content/drive/MyDrive/eph_ingresos/datos_originales/ipc True
/content/drive/MyDrive/eph_ingresos/datos_procesados True
/content/drive/MyDrive/eph_ingresos/documentacion True
/content/drive/MyDrive/eph_ingresos/resultados True


## 2. Librerías y versiones del entorno
Se importan todas las herramientas necesarias para el procesamiento y se registran las versiones utilizadas.

In [3]:
import sys
import os
import zipfile
import urllib.request
import requests
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import matplotlib as mpl
import seaborn as sns

print('Python', sys.version)
print('pandas', pd.__version__)
print('matplotlib', mpl.__version__)
print('seaborn', sns.__version__)

Python 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
pandas 2.2.3
matplotlib 3.10.0
seaborn 0.13.2


## 3. Descarga de las bases de la EPH
Se descargan los archivos comprimidos de la EPH desde el sitio oficial del INDEC, solo si no existen previamente en el almacenamiento. Se genera un registro con el estado de la descarga, el tamaño de cada archivo y su contenido.

In [4]:
base = "https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/"

urls = {
    2017: base + "EPH_usu_2_Trim_2017_txt.zip",
    2018: base + "EPH_usu_2_Trim_2018_txt.zip",
    2019: base + "EPH_usu_2_Trim_2019_txt.zip",
    2020: base + "EPH_usu_2_Trim_2020_txt.zip",
    2021: base + "EPH_usu_2_Trim_2021_txt.zip",
    2022: base + "EPH_usu_2_Trim_2022_txt.zip",
    2023: base + "EPH_usu_2_Trim_2023_txt.zip",
    2024: base + "EPH_usu_2_Trim_2024_txt.zip",
    2025: base + "EPH_usu_2_Trim_2025_txt.zip",
    2026: base + "EPH_usu_2_Trim_2026_txt.zip",
}

registro = []

for anio, url in urls.items():
    nombre_zip = url.split("/")[-1]
    ruta_zip = os.path.join(DATOS_EPH, nombre_zip)

    # Descarga solo si el archivo no está en Drive
    if os.path.exists(ruta_zip):
        estado = "ya existía"
    else:
        respuesta = requests.get(url, timeout=60)
        respuesta.raise_for_status()
        with open(ruta_zip, "wb") as f:
            f.write(respuesta.content)
        estado = "descargado"

    # Archivos internos del zip, sin descomprimir
    with zipfile.ZipFile(ruta_zip) as z:
        internos = z.namelist()

    # Fecha en que se escribió el archivo en Drive (hora UTC de Colab)
    fecha = datetime.fromtimestamp(os.path.getmtime(ruta_zip))

    registro.append({
        "anio": anio,
        "url": url,
        "archivo": nombre_zip,
        "bytes": os.path.getsize(ruta_zip),
        "fecha_descarga_utc": fecha.strftime("%Y-%m-%d %H:%M:%S"),
        "estado": estado,
        "archivos_internos": ", ".join(internos),
    })
    print(anio, estado)

df_registro = pd.DataFrame(registro)
df_registro.to_csv(DOCUMENTACION / "registro_descarga_eph.csv", index=False)
df_registro

2017 ya existía
2018 ya existía
2019 ya existía
2020 ya existía
2021 ya existía
2022 ya existía
2023 ya existía
2024 ya existía
2025 ya existía
2026 ya existía


,anio,url,archivo,bytes,fecha_descarga_utc,estado,archivos_internos
0,2017,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2017_txt.zip,3244184,2026-11-22 16:03:00,ya existía,"usu_individual_t217.txt, usu_hogar_t217.txt"
1,2018,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2018_txt.zip,2921791,2026-11-22 16:03:03,ya existía,"usu_individual_t218.txt, usu_hogar_t218.txt"
2,2019,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2019_txt.zip,3069931,2026-11-22 16:03:06,ya existía,"USU_individual_t219.txt, USU_hogar_t219.txt"
3,2020,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2020_txt.zip,1910484,2026-11-22 16:03:08,ya existía,"usu_Individual_T220.txt, usu_hogar_T220.txt"
4,2021,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2021_txt.zip,2578874,2026-11-22 16:03:11,ya existía,"usu_hogar_T221.txt, usu_individual_T221.txt"
5,2022,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2022_txt.zip,3137831,2026-11-22 16:03:14,ya existía,"EPH_usu_2do_Trim_2022_txt/, EPH_usu_2do_Trim_2..."
6,2023,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2023_txt.zip,2987879,2026-11-22 16:03:17,ya existía,"EPH_usu_2do_Trim_2023_txt/, EPH_usu_2do_Trim_2..."
7,2024,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2024_txt.zip,3622569,2026-11-22 16:03:19,ya existía,"EPH_usu_2_Trim_2024_txt/, EPH_usu_2_Trim_2024_..."
8,2025,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2025_txt.zip,3462812,2026-11-22 16:03:23,ya existía,"usu_individual_T225.txt, usu_hogar_T225.txt"
9,2026,https://www.indec.gob.ar/ftp/cuadros/menusuper...,EPH_usu_2_Trim_2026_txt.zip,2866599,2026-11-22 16:03:25,ya existía,"usu_hogar_T226.txt, usu_individual_T226.txt"


Se muestran los archivos internos de 2022 a 2024 para comprobar que sus nombres no son uniformes.

In [5]:
print(df_registro.loc[5:7, "archivos_internos"].tolist())

['EPH_usu_2do_Trim_2022_txt/, EPH_usu_2do_Trim_2022_txt/usu_hogar_T222.txt.txt, EPH_usu_2do_Trim_2022_txt/usu_individual_T222.txt.txt', 'EPH_usu_2do_Trim_2023_txt/, EPH_usu_2do_Trim_2023_txt/usu_hogar_T223.txt, EPH_usu_2do_Trim_2023_txt/usu_individual_T223.txt', 'EPH_usu_2_Trim_2024_txt/, EPH_usu_2_Trim_2024_txt/usu_hogar_T224.txt, EPH_usu_2_Trim_2024_txt/usu_individual_T224.txt']


## 4. Lectura y verificación de las bases

### 4.1 Función de lectura
Los nombres internos de los archivos zip no son uniformes: cambian las mayúsculas, hay subcarpetas en 2022–2024 y en 2022 la extensión es `.txt.txt`. Por eso la función busca el archivo por su contenido.

In [6]:
def leer_individual(anio):
    ruta_zip = os.path.join(DATOS_EPH, urls[anio].split("/")[-1])

    with zipfile.ZipFile(ruta_zip) as z:
        candidatos = [n for n in z.namelist() if "individual" in n.lower() and n.lower().endswith(".txt")]
        if len(candidatos) != 1:
            raise ValueError(f"{anio}: se esperaba 1 archivo individual, hay {candidatos}")
        nombre_ind = candidatos[0]
        df = pd.read_csv(z.open(nombre_ind), sep=";", low_memory=False)

    return df, nombre_ind

Se prueba la función con un año de nombre simple (2019) y otro con subcarpeta y doble extensión (2022).

In [7]:
for anio in [2019, 2022]:
    df_prueba, archivo = leer_individual(anio)
    print(anio, archivo, df_prueba.shape)

2019 USU_individual_t219.txt (59258, 177)
2022 EPH_usu_2do_Trim_2022_txt/usu_individual_T222.txt.txt (50614, 177)


### 4.2 Verificación de columnas y tipos
Se revisa que todas las columnas requeridas para el proyecto estén presentes y se validan sus tipos de datos en cada base anual.

In [8]:
COLUMNAS_PROYECTO = ["CODUSU", "CH06", "NIVEL_ED", "REGION", "ESTADO", "P21", "TOT_P12", "PONDIIO", "PONDERA"]

bases = {}
verificacion = []

for anio in urls:
    df_anio, archivo = leer_individual(anio)
    bases[anio] = df_anio

    faltantes = [c for c in COLUMNAS_PROYECTO if c not in df_anio.columns]

    verificacion.append({
        "anio": anio,
        "archivo": archivo,
        "filas": df_anio.shape[0],
        "columnas": df_anio.shape[1],
        "faltantes": ", ".join(faltantes),
        "tipo_P21": str(df_anio["P21"].dtype),
        "tipo_TOT_P12": str(df_anio["TOT_P12"].dtype),
        "tipo_PONDIIO": str(df_anio["PONDIIO"].dtype),
    })

df_verif = pd.DataFrame(verificacion)
df_verif.to_csv(DOCUMENTACION / "verificacion_bases_eph.csv", index=False)
df_verif

,anio,archivo,filas,columnas,faltantes,tipo_P21,tipo_TOT_P12,tipo_PONDIIO
0,2017,usu_individual_t217.txt,59755,177,,int64,int64,int64
1,2018,usu_individual_t218.txt,57835,177,,int64,int64,int64
2,2019,USU_individual_t219.txt,59258,177,,int64,int64,int64
3,2020,usu_Individual_T220.txt,37132,177,,int64,int64,int64
4,2021,usu_individual_T221.txt,47085,177,,int64,int64,int64
5,2022,EPH_usu_2do_Trim_2022_txt/usu_individual_T222....,50614,177,,int64,int64,int64
6,2023,EPH_usu_2do_Trim_2023_txt/usu_individual_T223.txt,49072,177,,int64,int64,int64
7,2024,EPH_usu_2_Trim_2024_txt/usu_individual_T224.txt,47151,235,,int64,int64,int64
8,2025,usu_individual_T225.txt,46086,235,,int64,int64,int64
9,2026,usu_individual_T226.txt,44467,234,,int64,int64,int64


Las nueve variables del proyecto están en los diez años, y `P21`, `TOT_P12` y `PONDIIO` son enteros. La cantidad de columnas pasa de 177 a 235 en 2024, porque el INDEC incorporó variables nuevas desde el 4.º trimestre de 2023, y a 234 en 2026, por la eliminación de `CAT_INAC`.

### 4.3 Comparación de columnas y categorías entre años
Se comprueba que las columnas y las categorías de las variables de agrupamiento sean las mismas en todos los años, para que los años puedan compararse entre sí. Las tablas muestran casos de la muestra, sin ponderar.

In [9]:
print(set(bases[2025].columns) - set(bases[2026].columns))
print(set(bases[2026].columns) - set(bases[2025].columns))

{'CAT_INAC'}
set()


In [10]:
tabla_nivel_ed = pd.DataFrame({anio: df["NIVEL_ED"].value_counts().sort_index() for anio, df in bases.items()})
tabla_nivel_ed

,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
NIVEL_ED,,,,,,,,,,
1,8882,8438,8751,5318,6773,7167,6922,6447,6205,5748
2,8191,7600,7380,4640,5517,5908,5439,5349,5067,5023
3,12402,11932,12401,7840,9876,10473,10365,9805,9449,8964
4,11159,10998,11321,7102,9349,10075,10125,10165,10418,10265
5,6844,6697,6953,4493,5759,6351,6123,5956,5912,5496
6,6822,6894,7287,4730,6287,6947,6600,6397,6410,6537
7,5455,5276,5165,3009,3524,3693,3498,3032,2625,2434


In [11]:
tabla_region = pd.DataFrame({anio: df["REGION"].value_counts().sort_index() for anio, df in bases.items()})
tabla_region

,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
REGION,,,,,,,,,,
1,11254,10587,10033,3935,3914,7463,7301,7275,7206,7072
40,12363,12327,12471,8725,11411,11043,10321,9955,10010,9661
41,5652,5621,6001,3971,4973,5147,5188,4932,4690,4559
42,5814,5510,5721,5158,5274,5243,5123,4730,4711,4352
43,17306,16767,17874,11442,14774,15411,15064,14524,14125,13700
44,7366,7023,7158,3901,6739,6307,6075,5735,5344,5123


In [12]:
tabla_estado = pd.DataFrame({anio: df["ESTADO"].value_counts().sort_index() for anio, df in bases.items()})
tabla_estado

,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
ESTADO,,,,,,,,,,
0,74,73,67,65,89,72,91,55,53,74
1,24238,23686,24365,12803,19408,22219,21797,20735,20374,19990
2,1860,1982,2405,1563,1737,1335,1279,1477,1387,1504
3,24771,23555,23869,17612,19765,20498,19620,19212,19193,18226
4,8812,8539,8552,5089,6086,6490,6285,5672,5079,4673


In [13]:
tabla_edad = pd.DataFrame({anio: {"min": df["CH06"].min(), "max": df["CH06"].max()} for anio, df in bases.items()})
tabla_edad

,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
min,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
max,103,104,102,102,102,104,100,102,103,102


**Lectura de los resultados:**
- **Columnas:** la única diferencia entre 2025 y 2026 es `CAT_INAC`, que el INDEC eliminó en 2026. El proyecto no la usa.
- **NIVEL_ED, REGION y ESTADO:** mantienen las mismas categorías en los diez años, así que no hace falta recodificarlas entre años. El código 9 de `NIVEL_ED` (sin dato) no aparece.
- **REGION:** la región GBA (código 1) cae de unos 10.000–11.000 casos en 2017–2019 a 3.935 en 2020 y 3.914 en 2021, por el relevamiento no presencial. Desde 2022 se mantiene en unos 7.000–7.500 casos.
- **CH06 (edad):** el mínimo −1 es el código de la EPH para los menores de un año. No afecta al proyecto, que trabaja con personas de 18 años o más.

## 5. Unión de los diez años y control de calidad
Se integran las bases anuales en un único conjunto de datos y se verifica la calidad general de los registros.

### 5.1 Unión y duplicados
Se apilan los diez años en una base única y se comprueba que no hay casos duplicados según los identificadores del hogar y la persona.

In [14]:
COLUMNAS_FINALES = ["ANO4", "CODUSU", "NRO_HOGAR", "COMPONENTE", "AGLOMERADO", "REGION",
                    "CH06", "NIVEL_ED", "ESTADO", "P21", "TOT_P12", "PONDIIO", "PONDERA"]

eph = pd.concat([df[COLUMNAS_FINALES] for anio, df in bases.items()], ignore_index=True)
print(eph.shape)
print(eph["ANO4"].value_counts().sort_index())
print(eph.duplicated(subset=["ANO4", "CODUSU", "NRO_HOGAR", "COMPONENTE"]).sum())

(498455, 13)
ANO4
2017    59755
2018    57835
2019    59258
2020    37132
2021    47085
2022    50614
2023    49072
2024    47151
2025    46086
2026    44467
Name: count, dtype: int64
0


### 5.2 Faltantes y códigos especiales
Se verifica que no hay valores vacíos (NaN) en las variables seleccionadas y que el valor −9 (no respuesta) es el único código negativo presente en los ingresos `P21` y `TOT_P12`.

In [15]:
eph.isna().sum()

,0
ANO4,0
CODUSU,0
NRO_HOGAR,0
COMPONENTE,0
AGLOMERADO,0
REGION,0
CH06,0
NIVEL_ED,0
ESTADO,0
P21,0


In [16]:
print(eph.loc[eph["P21"] < 0, "P21"].unique())
print(eph.loc[eph["TOT_P12"] < 0, "TOT_P12"].unique())

[-9]
[-9]


### 5.3 No respuesta del ingreso y ponderador PONDIIO
Se analiza la relación entre la falta de respuesta en el ingreso de la ocupación principal (`P21` = −9) y el ponderador individual para ingresos (`PONDIIO`). Un hallazgo central es que todos los casos con `P21` = −9 tienen `PONDIIO` = 0, y todos los demás tienen `PONDIIO` > 0. Esto significa que el diseño del INDEC ya excluye a los no respondentes al calcular estadísticos de ingresos.

In [17]:
ocupados = eph[eph["ESTADO"] == 1]
pd.crosstab(ocupados["ANO4"], np.sign(ocupados["P21"]))

P21,-1,0,1
ANO4,,,
2017,3889,645,19704
2018,3418,674,19594
2019,2634,671,21060
2020,1446,712,10645
2021,2723,563,16122
2022,3557,546,18116
2023,3713,456,17628
2024,3621,400,16714
2025,3665,449,16260


In [18]:
pd.crosstab(np.sign(ocupados["P21"]), ocupados["PONDIIO"] == 0)

PONDIIO,False,True
P21,,
-1,0,32358
0,5485,0
1,171772,0


## 6. Población de estudio y variables derivadas
Se define la población de estudio y se crean las variables de agrupamiento.

### 6.1 Personas sin instrucción
Se cuentan los adultos sin instrucción (`NIVEL_ED` = 7) en cada año para decidir cómo tratarlos.

In [19]:
eph_sin_instruccion = eph[(eph["CH06"] >= 18) & (eph["NIVEL_ED"] == 7)]
print(eph_sin_instruccion["ANO4"].value_counts().sort_index())

ANO4
2017    365
2018    355
2019    357
2020    191
2021    248
2022    281
2023    271
2024    233
2025    199
2026    212
Name: count, dtype: int64


Son entre 191 y 365 adultos por año, demasiado pocos para un grupo propio. Por eso se agrupan con "Primario incompleto".

### 6.2 Grupos de edad y nivel educativo
Se filtra la base para conservar únicamente a las personas de 18 años o más. Después se agrupa la edad (18–29, 30–39, 40–64 y 65 o más) y se recategoriza el nivel educativo en primario, secundario y superior, cada uno incompleto o completo.

In [20]:
eph_18 = eph[eph["CH06"] >= 18].copy()

eph_18["grupo_edad"] = pd.cut(
    eph_18["CH06"],
    bins=[17, 29, 39, 64, np.inf],
    labels=["18-29", "30-39", "40-64", "65 o más"],
)

NIVELES = {
    1: "Primario incompleto",
    7: "Primario incompleto",   # sin instrucción se agrupa con primario incompleto (pocos casos)
    2: "Primario completo",
    3: "Secundario incompleto",
    4: "Secundario completo",
    5: "Superior incompleto",
    6: "Superior completo",
}
eph_18["nivel_educativo"] = eph_18["NIVEL_ED"].map(NIVELES)

print(eph_18[["grupo_edad", "nivel_educativo"]].isna().sum())
print(pd.crosstab(eph_18["ANO4"], eph_18["grupo_edad"]))

grupo_edad         0
nivel_educativo    0
dtype: int64
grupo_edad  18-29  30-39  40-64  65 o más
ANO4                                     
2017        11760   8089  16189      7192
2018        11329   7896  15776      7058
2019        11604   8083  16312      7160
2020         7219   4930  10525      4688
2021         8932   6405  13490      6185
2022         9534   7129  14443      6676
2023         9340   6981  14146      6063
2024         9039   6559  13729      6214
2025         8425   6637  13638      6445
2026         8182   6300  13307      6337


### 6.3 Etapa y marca de relevamiento no presencial
Se marca el período de análisis, destacando que 2020 y 2021 se relevaron de forma no presencial.

### 6.4 Guardado de la base de adultos
Se exporta la base unificada y filtrada para su uso en las siguientes etapas.

In [21]:
eph_18["etapa"] = pd.cut(
    eph_18["ANO4"],
    bins=[2016, 2019, 2021, 2026],
    labels=["antes", "durante", "después"],
)
eph_18["no_presencial"] = eph_18["ANO4"].isin([2020, 2021])

RUTA_SALIDA = DATOS_PROCESADOS / "eph_t2_2017_2026_18mas.parquet"
eph_18.to_parquet(RUTA_SALIDA, index=False)

control = pd.read_parquet(RUTA_SALIDA)
print(control.shape)
print(control.dtypes)

(369946, 17)
ANO4                  int64
CODUSU               object
NRO_HOGAR             int64
COMPONENTE            int64
AGLOMERADO            int64
REGION                int64
CH06                  int64
NIVEL_ED              int64
ESTADO                int64
P21                   int64
TOT_P12               int64
PONDIIO               int64
PONDERA               int64
grupo_edad         category
nivel_educativo      object
etapa              category
no_presencial          bool
dtype: object


## 7. Situación del ingreso entre los adultos ocupados
Muestra, para cada año, cuántos ocupados no respondieron el ingreso (−9), cuántos declararon cero y cuántos declararon un monto positivo, tanto en P21 como en TOT_P12. Son casos de la muestra, sin ponderar.

In [22]:
# La Etapa 3 parte de la salida documentada de la Etapa 2: adultos de 18 años o más
adultos = pd.read_parquet(DATOS_PROCESADOS / "eph_t2_2017_2026_18mas.parquet")

# Controles: deben coincidir con el registro de la Etapa 2
print(len(adultos))           # 369946
print(adultos["CH06"].min())  # 18



369946
18


In [23]:
# Solo adultos ocupados (ESTADO = 1). .copy() evita la advertencia al crear columnas.
ocup = adultos[adultos["ESTADO"] == 1].copy()

# P21: tres situaciones excluyentes
ocup["p21_nr"]   = ocup["P21"].eq(-9)   # no respondió
ocup["p21_cero"] = ocup["P21"].eq(0)    # declaró ingreso cero
ocup["p21_pos"]  = ocup["P21"].gt(0)    # declaró un monto positivo

# TOT_P12: no respuesta y casos con ingreso de otras ocupaciones
ocup["tot_nr"]  = ocup["TOT_P12"].eq(-9)
ocup["tot_pos"] = ocup["TOT_P12"].gt(0)

len(ocup)

208828

In [24]:
# Casos de la muestra, sin ponderar: sum() cuenta los True de cada columna
resumen = ocup.groupby("ANO4").agg(
    ocupados=("P21", "size"),
    p21_nr=("p21_nr", "sum"),
    p21_cero=("p21_cero", "sum"),
    p21_pos=("p21_pos", "sum"),
    tot_nr=("tot_nr", "sum"),
    tot_pos=("tot_pos", "sum"),
)

# Las tres situaciones de P21 deben sumar el total de ocupados en todos los años
cubre_todo = resumen[["p21_nr", "p21_cero", "p21_pos"]].sum(axis=1).eq(resumen["ocupados"]).all()
print("P21 cubre a todos los ocupados:", cubre_todo)

resumen

P21 cubre a todos los ocupados: True


,ocupados,p21_nr,p21_cero,p21_pos,tot_nr,tot_pos
ANO4,,,,,,
2017,24127,3877,618,19632,346,1693
2018,23571,3412,642,19517,332,1843
2019,24247,2626,636,20985,318,2263
2020,12770,1443,702,10625,134,829
2021,19326,2717,537,16072,275,1500
2022,22141,3553,530,18058,341,1980
2023,21740,3704,446,17590,373,1981
2024,20660,3611,390,16659,369,1897
2025,20312,3651,441,16220,369,1895


In [25]:
# Porcentaje sobre el total de ocupados de cada año
porcentajes = (resumen.drop(columns="ocupados")
               .div(resumen["ocupados"], axis=0) * 100).round(1)
display(porcentajes)

# Entre quienes respondieron P21, ¿cuántos no respondieron TOT_P12?
respondieron = ocup[ocup["P21"] >= 0]
respondieron.groupby("ANO4")["tot_nr"].sum()

,p21_nr,p21_cero,p21_pos,tot_nr,tot_pos
ANO4,,,,,
2017,16.1,2.6,81.4,1.4,7.0
2018,14.5,2.7,82.8,1.4,7.8
2019,10.8,2.6,86.5,1.3,9.3
2020,11.3,5.5,83.2,1.0,6.5
2021,14.1,2.8,83.2,1.4,7.8
2022,16.0,2.4,81.6,1.5,8.9
2023,17.0,2.1,80.9,1.7,9.1
2024,17.5,1.9,80.6,1.8,9.2
2025,18.0,2.2,79.9,1.8,9.3


,tot_nr
ANO4,
2017,146
2018,130
2019,133
2020,42
2021,87
2022,124
2023,116
2024,137
2025,129


Se observa una forma de U en la no respuesta de P21 (16,1 % en 2017, mínimo de 10,8 % en 2019, 18,5 % en 2026) y la duplicación del ingreso cero en 2020.

## 8. Decisiones sobre los ingresos
- **Indicador central:** `p21_real`, solo con P21 > 0 y ponderador PONDIIO. La proporción de ocupados con ingreso cero se informa aparte.
- **Indicador ampliado:** `ing_lab_real` = P21 + TOT_P12, con PONDIIO. Se excluyen además los casos con TOT_P12 = −9. Se descartó PONDII porque corrige la no respuesta del ingreso total individual, que incluye ingresos no laborales.
- **Limitación:** TOT_P12 puede incluir pagos retroactivos, que no se pueden separar.

## 9. IPC nacional y deflactor
- La serie es el IPC nacional, con base diciembre de 2016 = 100, sin cambios de base hasta 2026.
- El mes base es octubre de 2026 y queda fijo.
- El deflactor de cada año es el promedio del IPC de marzo a mayo, porque la EPH pregunta por el ingreso del mes anterior a la entrevista. Es una aproximación.
- El factor indica cuántos pesos de octubre de 2026 equivalen a un peso del año.

### 9.1 Descarga del IPC
El archivo se guarda en `datos_originales/ipc/` y la descarga se registra en `documentacion/registro_descarga_ipc.csv`. Si el archivo ya está en Drive, no se descarga de nuevo, para no sobrescribir el original.

In [26]:
URL_IPC = "https://www.indec.gob.ar/ftp/cuadros/economia/serie_ipc_divisiones.csv"
archivo_ipc = DATOS_IPC / "serie_ipc_divisiones.csv"

if archivo_ipc.exists():
    # No sobrescribir el archivo original si ya existe
    registro_ipc = pd.read_csv(DOCUMENTACION / "registro_descarga_ipc.csv")
else:
    urllib.request.urlretrieve(URL_IPC, archivo_ipc)
    momento = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires"))
    registro_ipc = pd.DataFrame([{
        "url": URL_IPC,
        "archivo": archivo_ipc.name,
        "bytes": archivo_ipc.stat().st_size,
        "descarga": momento.strftime("%d/%m/%Y %H:%M"),
    }])
    registro_ipc.to_csv(DOCUMENTACION / "registro_descarga_ipc.csv", index=False)

registro_ipc

,url,archivo,bytes,descarga
0,https://www.indec.gob.ar/ftp/cuadros/economia/...,serie_ipc_divisiones.csv,1252806,23/11/2026 12:08


### 9.2 Lectura y selección de la serie
La primera lectura muestra `Indice_IPC` como texto por la coma decimal; la segunda lo corrige con `decimal=","`.

In [27]:
ipc_crudo = pd.read_csv(archivo_ipc, sep=";", encoding="latin-1")

print(ipc_crudo.columns.tolist())
print(ipc_crudo.dtypes)
ipc_crudo.head()

['Codigo', 'Descripcion', 'Clasificador', 'Periodo', 'Indice_IPC', 'v_m_IPC', 'v_i_a_IPC', 'Region']
Codigo          object
Descripcion     object
Clasificador    object
Periodo          int64
Indice_IPC      object
v_m_IPC         object
v_i_a_IPC       object
Region          object
dtype: object


,Codigo,Descripcion,Clasificador,Periodo,Indice_IPC,v_m_IPC,v_i_a_IPC,Region
0,0,NIVEL GENERAL,Nivel general y divisiones COICOP,201612,100,NaN,NaN,GBA
1,0,NIVEL GENERAL,Nivel general y divisiones COICOP,201612,100,NaN,NaN,Pampeana
2,0,NIVEL GENERAL,Nivel general y divisiones COICOP,201612,100,NaN,NaN,Noreste
3,0,NIVEL GENERAL,Nivel general y divisiones COICOP,201612,100,NaN,NaN,Noroeste
4,0,NIVEL GENERAL,Nivel general y divisiones COICOP,201612,100,NaN,NaN,Cuyo


In [28]:
ipc_crudo = pd.read_csv(archivo_ipc, sep=";", encoding="latin-1", decimal=",")

print(ipc_crudo.dtypes["Indice_IPC"])
print(ipc_crudo["Region"].unique())

float64
['GBA' 'Pampeana' 'Noreste' 'Noroeste' 'Cuyo' 'Patagonia' 'Nacional']


In [29]:
# Nivel general del total del país: dos condiciones a la vez
ipc_nac = ipc_crudo[(ipc_crudo["Region"] == "Nacional") &
                    (ipc_crudo["Descripcion"] == "NIVEL GENERAL")].copy()

print(len(ipc_nac))                    # 119
print(ipc_nac["Periodo"].is_unique)    # True

# Índice en el mes base de la serie y en el último mes
ipc_nac.loc[ipc_nac["Periodo"].isin([201612, 202610]), ["Periodo", "Indice_IPC"]]

119
True


,Periodo,Indice_IPC
6,201612,100.0000
14874,202610,12785.9321


### 9.3 Cálculo del factor de cada año
Fórmula del ajuste:
$$ factor_t = \frac{\text{IPC de octubre de 2026}}{\text{promedio del IPC de marzo a mayo del año } t} $$

In [30]:
# 1. Año y mes a partir de Periodo (AAAAMM)
ipc_nac["anio"] = ipc_nac["Periodo"] // 100   # 201703 // 100 = 2017
ipc_nac["mes"]  = ipc_nac["Periodo"] % 100    # 201703 % 100  = 3

# 2. Meses de referencia del ingreso del 2.º trimestre (marzo a mayo), sin 2016
ref = ipc_nac[ipc_nac["mes"].isin([3, 4, 5]) & (ipc_nac["anio"] >= 2017)]

# 3. Promedio del IPC de marzo a mayo de cada año
ipc_prom = ref.groupby("anio")["Indice_IPC"].mean()

# 4. IPC del mes base (octubre de 2026), tomado de la serie
ipc_base = ipc_nac.loc[ipc_nac["Periodo"] == 202610, "Indice_IPC"].iloc[0]

# 5. Factor de cada año: cuántos pesos de oct-2026 vale un peso de ese año
factor = ipc_base / ipc_prom

factor

,Indice_IPC
anio,
2017,117.790437
2018,93.682209
2019,60.067529
2020,41.219036
2021,28.240788
2022,17.872894
2023,8.539203
2024,2.222219
2025,1.495232


## 10. Ingresos en pesos de octubre de 2026
Se aplican los factores de ajuste a los ingresos. El método `.where()` deja NaN en los casos no válidos, así que el −9 nunca se multiplica por el factor. Luego se realizan controles para confirmar que ningún año quede sin factor asignado y que no existan ingresos reales negativos o iguales a cero.

In [31]:
factor = factor.rename("factor_ipc")

# 1. Factor de cada año, asignado a cada persona según su ANO4
adultos["factor_ipc"] = adultos["ANO4"].map(factor)

# 2. Ingreso de la ocupación principal en pesos de oct-2026.
#    .where() deja NaN donde la condición es falsa: el −9 y el 0 no se deflactan.
valido_p21 = adultos["P21"] > 0
adultos["p21_real"] = (adultos["P21"] * adultos["factor_ipc"]).where(valido_p21)

# 3. Ingreso laboral ampliado: P21 válido y TOT_P12 respondido (0 o más)
valido_amp = valido_p21 & (adultos["TOT_P12"] >= 0)
adultos["ing_lab_real"] = ((adultos["P21"] + adultos["TOT_P12"])
                           * adultos["factor_ipc"]).where(valido_amp)

In [32]:
# Ningún año quedó sin factor
print("Sin factor:", adultos["factor_ipc"].isna().sum())                      # 0

# No hay ingresos reales negativos ni iguales a cero
print("Mínimo p21_real:", adultos["p21_real"].min())                          # > 0

# Los casos con p21_real coinciden con los p21_pos de la sección 7
print("Coincide con la sección 7:",
      adultos.groupby("ANO4")["p21_real"].count().eq(resumen["p21_pos"]).all())  # True

# El ampliado nunca tiene más casos que el central
adultos.groupby("ANO4")[["p21_real", "ing_lab_real"]].count()

Sin factor: 0
Mínimo p21_real: 3415.681313406699
Coincide con la sección 7: True


,p21_real,ing_lab_real
ANO4,,
2017,19632,19490
2018,19517,19396
2019,20985,20856
2020,10625,10587
2021,16072,15993
2022,18058,17942
2023,17590,17480
2024,16659,16530
2025,16220,16093


## 11. Guardado de las bases finales
Se exportan los datos procesados.

In [33]:
# Base nueva: la salida de la Etapa 2 no se modifica
adultos.to_parquet(DATOS_PROCESADOS / "eph_t2_2017_2026_18mas_ingresos.parquet", index=False)

# Deflactor aparte, para poder revisar el cálculo
factor.to_frame().to_csv(DATOS_PROCESADOS / "deflactor_ipc_t2.csv")

(DATOS_PROCESADOS / "eph_t2_2017_2026_18mas_ingresos.parquet").exists(), (DATOS_PROCESADOS / "deflactor_ipc_t2.csv").exists()

(True, True)

## 12. Resumen y limitaciones

**Cifras principales:** 369.946 adultos de 18 años o más en los diez años. La base final tiene 20 columnas y agrega `factor_ipc`, `p21_real` e `ing_lab_real`, en pesos de octubre de 2026.

**Archivos generados:**
1. `datos_originales/eph/*.zip`
2. `documentacion/registro_descarga_eph.csv`
3. `documentacion/verificacion_bases_eph.csv`
4. `datos_procesados/eph_t2_2017_2026_18mas.parquet`
5. `datos_originales/ipc/serie_ipc_divisiones.csv`
6. `documentacion/registro_descarga_ipc.csv`
7. `datos_procesados/deflactor_ipc_t2.csv`
8. `datos_procesados/eph_t2_2017_2026_18mas_ingresos.parquet`

**Limitaciones:**
- **Relevamiento no presencial en 2020 y 2021:** las cifras de esos años combinan el efecto de la pandemia con el del cambio de modalidad, y los dos no se pueden separar.
- **Región GBA:** tiene alrededor de 3.900 casos por año en 2020 y 2021, frente a unos 10.600 en 2017–2019.
- **No respuesta del ingreso:** es alta y crece desde 2019, hasta el 18,5 % de los ocupados en 2026.
- **Base preliminar:** la del 2.º trimestre de 2026 puede ser reemplazada por una versión definitiva.
- **Deflactor:** el promedio del IPC de marzo a mayo es una aproximación al mes de referencia del ingreso.
- **Ingreso ampliado:** `TOT_P12`, el ingreso de otras ocupaciones, puede incluir pagos retroactivos que no se pueden separar.
- **Cobertura:** los resultados representan a los 31 aglomerados urbanos de la EPH, no a la población rural ni al total del país.

**Próximo paso:** `02_analisis.ipynb` parte de `datos_procesados/eph_t2_2017_2026_18mas_ingresos.parquet`.